In [1]:
"""
Análisis exploratorio de telemetría por finca
=============================================

Análisis incluidos (cada uno es una función que recibe una `Finca`):

    1. analisis_boxplots_diarios(finca)       -> faltantes, outliers (z-score) y boxplots diarios
    2. analisis_tendencias_diarias(finca)     -> promedio ± desviación diaria de todas las variables
    3. analisis_violin_franjas(finca)         -> estadísticas y violin plots por franja horaria
    4. analisis_series_por_franja(finca)      -> series temporales diarias por franja horaria

Uso rápido:
    ejecutar_todo()                                   # todos los análisis, todas las fincas
    ejecutar_todo(analisis=["violin_franjas"])        # solo algunos análisis
    analisis_boxplots_diarios(FINCAS[0])              # un análisis, una finca
"""

import os
import warnings
from dataclasses import dataclass

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# =====================================================================
# 1. CONFIGURACIÓN GLOBAL
# =====================================================================
@dataclass(frozen=True)
class Finca:
    nombre: str
    fecha_inicio: str
    fecha_fin: str


# Lista de fincas con su rango de análisis (agregar aquí nuevas fincas)
FINCAS = [
    Finca("La_Orquidea", "2025-12-09", "2026-05-21"),
    # Finca("Otra_Finca", "YYYY-MM-DD", "YYYY-MM-DD"),
]

ROOT_BD = R"D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Nuevos-Modelos-Paulina\Datos"
DIR_SALIDA = "outputs"

# Formato del archivo: telemetria_{tipo}_{finca}_{RANGO_ARCHIVOS}.xlsx
RANGO_ARCHIVOS = "2024-01-01_to_2026-10-19"

# Clave usada en las gráficas -> tipo en el nombre del archivo
ARCHIVOS_TELEMETRIA = {
    "actividad": "actividad",
    "diarias": "variables_diarias",
    "fija": "fija",
    "sistema": "sistema",
    "movil": "movil",
}

COLUMNAS_COORDENADAS = {"lat", "lon", "latitud", "longitud"}
COLUMNAS_GEOGRAFICAS = ["departamento", "municipio", "tipo", "nombre", "lat", "lon"]

# Franjas horarias y colores (azul: mañana, naranja: tarde, verde: noche)
COLORES_FRANJA = {
    "1. Mañana (6am-12m)": "#4c72b0",
    "2. Tarde (12m-6pm)": "#dd8452",
    "3. Noche (6pm-6am)": "#55a868",
}
FRANJAS = list(COLORES_FRANJA.keys())


In [2]:


# =====================================================================
# 2. UTILIDADES COMUNES
# =====================================================================
def configurar_estilo(font_scale):
    """Estilo científico de las gráficas (el font_scale varía por análisis)."""
    plt.style.use("seaborn-v0_8-whitegrid")
    sns.set_context("paper", font_scale=font_scale)
    plt.rcParams["font.family"] = "serif"


def ruta_telemetria(finca, tipo):
    return f"{ROOT_BD}/{finca.nombre}/telemetria_{tipo}_{finca.nombre}_{RANGO_ARCHIVOS}.xlsx"


def cargar_telemetria(finca, tipo, col_fecha=None):
    """
    Lee un Excel de telemetría, filtra por el rango de fechas de la finca
    y deja la fecha como índice. Si `col_fecha` es None, la detecta
    (columna llamada 'timestamp' o 'fecha').
    """
    df = pd.read_excel(ruta_telemetria(finca, tipo))
    if col_fecha is None:
        col_fecha = [c for c in df.columns if c.lower() in ["timestamp", "fecha"]][0]
    df[col_fecha] = pd.to_datetime(df[col_fecha])
    df = df[(df[col_fecha] >= finca.fecha_inicio) & (df[col_fecha] <= finca.fecha_fin)].copy()
    df.set_index(col_fecha, inplace=True)
    return df


def columnas_numericas_validas(df):
    """Numéricas que no están vacías al 100% y que no son coordenadas."""
    cols = df.select_dtypes(include=[np.number]).dropna(axis=1, how="all").columns
    return [c for c in cols if c.lower() not in COLUMNAS_COORDENADAS]


def sin_columnas_geograficas(df):
    return df.drop(columns=[c for c in COLUMNAS_GEOGRAFICAS if c in df.columns])


def nombres_numericos(df):
    return df.select_dtypes(include=[np.number]).columns.tolist()


def asignar_franja(hora):
    """Clasifica una hora entera en tres segmentos del día."""
    if 6 <= hora < 12:
        return "1. Mañana (6am-12m)"
    elif 12 <= hora < 18:
        return "2. Tarde (12m-6pm)"
    else:
        return "3. Noche (6pm-6am)"


def crear_directorio(finca, subcarpeta):
    ruta = f"{DIR_SALIDA}/{finca.nombre}/{subcarpeta}"
    os.makedirs(ruta, exist_ok=True)
    return ruta


def mostrar_tabla(df):
    """display() en Jupyter; print() fuera de Jupyter."""
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df)


# =====================================================================
# 3. ANÁLISIS 1: FALTANTES, OUTLIERS Y BOXPLOTS DIARIOS
# =====================================================================
def tabla_datos_faltantes(df):
    df_missing = df.isnull().sum().reset_index()
    df_missing.columns = ["Variable", "Datos_Faltantes"]
    df_missing["Porcentaje_%"] = (df_missing["Datos_Faltantes"] / len(df)) * 100
    return df_missing[df_missing["Datos_Faltantes"] > 0]


def tabla_outliers_zscore(df_fija, df_sist, cols_fija, cols_sist, umbral=3):
    """Outliers (|z| > umbral) sobre promedios horarios de ambas telemetrías."""
    df_h_fija = df_fija[cols_fija].resample("h").mean()
    df_h_sist = df_sist[cols_sist].resample("h").mean()
    df_h = df_h_fija.join(df_h_sist)

    outliers = []
    for var in list(cols_fija) + list(cols_sist):
        if var in df_h.columns:
            mean_v = df_h[var].mean()
            std_v = df_h[var].std()
            if pd.notnull(mean_v) and std_v > 0:
                z = np.abs((df_h[var] - mean_v) / std_v)
                out = df_h[z > umbral]
                for idx, row in out.iterrows():
                    outliers.append({
                        "Fecha_Hora": idx.strftime("%Y-%m-%d %H:%M"),
                        "Variable": var,
                        "Valor_Atipico": round(row[var], 2),
                        "Voltaje_Bateria_V": round(row.get("mppt_battery_voltage_V", np.nan), 2),
                    })
    return pd.DataFrame(outliers)


def graficar_boxplots_diarios(finca, variables, dir_salida, chunk_size=8):
    """
    `variables`: lista de tuplas (dataframe, nombre_variable).
    Se procesa en bloques para no saturar memoria con imágenes muy largas.
    """
    paleta = sns.color_palette("husl", len(variables))

    for start in range(0, len(variables), chunk_size):
        chunk = variables[start:start + chunk_size]

        fig, axes = plt.subplots(len(chunk), 1, figsize=(38, 6 * len(chunk)))
        if len(chunk) == 1:
            axes = [axes]

        for i, (df, var) in enumerate(chunk):
            ax = axes[i]
            df_plot = df[["Date", var]].dropna()

            sns.boxplot(
                data=df_plot,
                x="Date",
                y=var,
                ax=ax,
                color=paleta[start + i],
                linewidth=1.5,
                fliersize=3,
                showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white",
                           "markeredgecolor": "black", "markersize": "5"},
            )

            ax.set_title(f"Distribución Diaria (Boxplot): {var} - {finca.nombre}",
                         fontweight="bold", fontsize=22)
            ax.set_ylabel(f"Valor ({var})", fontsize=18)
            ax.set_xlabel("Fecha", fontsize=18)

            # Etiquetas del eje X espaciadas según la densidad
            xticks = ax.get_xticks()
            xticklabels = [item.get_text() for item in ax.get_xticklabels()]
            step = max(1, len(xticks) // 40)
            ax.set_xticks(xticks[::step])
            ax.set_xticklabels(xticklabels[::step], rotation=45, ha="right", fontsize=14)

            ax.grid(axis="y", linestyle="--", alpha=0.7)
            ax.grid(axis="x", linestyle=":", alpha=0.4)

        plt.tight_layout()

        ruta = f"{dir_salida}/boxplots_vars_{start + 1}_a_{start + len(chunk)}.png"
        fig.savefig(ruta, dpi=500, bbox_inches="tight")
        print(f"Bloque {start + 1}-{start + len(chunk)} guardado en {ruta}")
        plt.close(fig)  # libera memoria

        plt.show()


def analisis_boxplots_diarios(finca, chunk_size=8):
    configurar_estilo(1.5)

    df_fija = cargar_telemetria(finca, "fija", col_fecha="timestamp")
    df_sist = cargar_telemetria(finca, "sistema", col_fecha="timestamp")

    # Componente de fecha (día) para agrupar los boxplots
    df_fija["Date"] = df_fija.index.date
    df_sist["Date"] = df_sist.index.date

    cols_fija = columnas_numericas_validas(df_fija)
    cols_sist = columnas_numericas_validas(df_sist)

    # Tablas
    df_missing = tabla_datos_faltantes(df_fija)
    df_outliers = tabla_outliers_zscore(df_fija, df_sist, cols_fija, cols_sist)

    print("=== DATAFRAME: VARIABLES CON DATOS FALTANTES ===")
    mostrar_tabla(df_missing)
    print("\n=== DATAFRAME: MUESTRA DE OUTLIERS ===")
    mostrar_tabla(df_outliers.head(10))

    # Boxplots
    variables = [(df_fija, v) for v in cols_fija] + [(df_sist, v) for v in cols_sist]
    dir_salida = crear_directorio(finca, "boxplots_diarios")
    graficar_boxplots_diarios(finca, variables, dir_salida, chunk_size)


# =====================================================================
# 4. ANÁLISIS 2: TENDENCIAS DIARIAS DE TODAS LAS VARIABLES
# =====================================================================
def cargar_todas_las_telemetrias(finca):
    """Diccionario {clave: dataframe solo con columnas numéricas válidas}."""
    dfs = {}
    for key, tipo in ARCHIVOS_TELEMETRIA.items():
        try:
            df_temp = cargar_telemetria(finca, tipo)
            cols_num = columnas_numericas_validas(df_temp)
            if len(cols_num) > 0:
                dfs[key] = df_temp[cols_num]
        except Exception as e:
            print(f"Nota: No se pudo procesar el archivo {key} o no contiene datos en este rango de fechas. ({e})")
    return dfs


def promedio_y_std_diarios(dfs):
    """Une todos los datasets con medias y desviaciones diarias (columnas 'var (clave)')."""
    medias, desvs = [], []
    for key, df in dfs.items():
        df_mean = df.resample("D").mean()
        df_mean.columns = [f"{c} ({key})" for c in df_mean.columns]
        medias.append(df_mean)

        df_std = df.resample("D").std()
        df_std.columns = [f"{c} ({key})" for c in df_std.columns]
        desvs.append(df_std)

    return pd.concat(medias, axis=1), pd.concat(desvs, axis=1)


def graficar_tendencias(finca, df_all_mean, df_all_std, dir_salida, chunk_size=10):
    variables_totales = df_all_mean.columns.tolist()

    for start in range(0, len(variables_totales), chunk_size):
        chunk_vars = variables_totales[start:start + chunk_size]

        fig, axes = plt.subplots(len(chunk_vars), 1, figsize=(20, 3.5 * len(chunk_vars)))
        if len(chunk_vars) == 1:
            axes = [axes]

        for i, var in enumerate(chunk_vars):
            ax = axes[i]
            mean_vals = df_all_mean[var]
            std_vals = df_all_std[var].fillna(0)  # 0 si no hay datos suficientes para calcular std

            ax.plot(mean_vals.index, mean_vals, color="#1f77b4", linewidth=2, label="Promedio Diario")
            ax.fill_between(mean_vals.index,
                            mean_vals - std_vals,
                            mean_vals + std_vals,
                            color="#1f77b4", alpha=0.3, label="Desviación Estándar")

            ax.set_title(f"Evolución Diaria: {var} - {finca.nombre}", fontweight="bold", fontsize=14)
            ax.set_ylabel(var.split(" (")[0], fontsize=12)
            ax.legend(loc="upper right")
            ax.grid(axis="both", linestyle="--", alpha=0.6)
            ax.tick_params(axis="x", rotation=45)

        plt.tight_layout()

        ruta = f"{dir_salida}/tendencias_bloque_{start + 1}_a_{start + len(chunk_vars)}.png"
        fig.savefig(ruta, dpi=500, bbox_inches="tight")

        plt.show()


def analisis_tendencias_diarias(finca, chunk_size=10):
    configurar_estilo(1.2)

    dfs = cargar_todas_las_telemetrias(finca)
    df_all_mean, df_all_std = promedio_y_std_diarios(dfs)
    print(f"Total de variables numéricas a graficar: {len(df_all_mean.columns)}")

    dir_salida = crear_directorio(finca, "tendencias_todas_las_variables")
    graficar_tendencias(finca, df_all_mean, df_all_std, dir_salida, chunk_size)


# =====================================================================
# 5. ANÁLISIS 3: DISTRIBUCIÓN POR FRANJA HORARIA (VIOLIN PLOTS)
# =====================================================================
def analisis_violin_franjas(finca):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        configurar_estilo(1.3)

        # Telemetría fija y de sistema: correlaciona variables de suelo con energía
        df_fija = cargar_telemetria(finca, "fija", col_fecha="timestamp")
        df_sist = cargar_telemetria(finca, "sistema", col_fecha="timestamp")

        # Franja horaria
        df_fija["Franja_Horaria"] = df_fija.index.hour.map(asignar_franja)
        df_sist["Franja_Horaria"] = df_sist.index.hour.map(asignar_franja)

        vars_fija = nombres_numericos(sin_columnas_geograficas(df_fija))
        vars_sist = nombres_numericos(sin_columnas_geograficas(df_sist))

        # Inner join: mismo instante temporal en ambas telemetrías
        df_combinado = df_fija[vars_fija + ["Franja_Horaria"]].join(df_sist[vars_sist], how="inner")

        # Resumen estadístico tabular
        print("\n=== RESUMEN ESTADÍSTICO POR FRANJA HORARIA ===")
        resumen_stats = df_combinado.groupby("Franja_Horaria").agg(["mean", "std", "min", "max"]).round(2)
        print(resumen_stats.to_string())

        dir_salida = crear_directorio(finca, "analisis_horario")
        resumen_stats.to_excel(f"{dir_salida}/estadisticas_franjas_horarias_{finca.nombre}.xlsx")

        # Violin plots (2 columnas)
        variables_plot = vars_fija + vars_sist
        n_filas = (len(variables_plot) + 1) // 2
        fig, axes = plt.subplots(n_filas, 2, figsize=(20, 5 * n_filas))
        axes = axes.flatten()

        for i, var in enumerate(variables_plot):
            sns.violinplot(
                data=df_combinado,
                x="Franja_Horaria",
                y=var,
                ax=axes[i],
                palette=list(COLORES_FRANJA.values()),
                inner="quartile",  # cuartiles 25, 50 (mediana) y 75
                linewidth=1.2,
            )
            axes[i].set_title(f"Distribución de {var}\nsegún Franja", fontweight="bold", fontsize=15)
            axes[i].set_xlabel("")
            axes[i].set_ylabel(var, fontsize=12)
            axes[i].tick_params(axis="x", rotation=15)

        plt.suptitle(
            f"Análisis Distribucional por Franja Horaria - {finca.nombre}\n"
            f"({finca.fecha_inicio} a {finca.fecha_fin})",
            fontsize=20, fontweight="bold", y=1.02,
        )
        plt.tight_layout()

        ruta = f"{dir_salida}/violin_plots_horarios_{finca.nombre}.png"
        fig.savefig(ruta, dpi=200, bbox_inches="tight")
        print(f"\nGráfica exportada exitosamente a: {ruta}")

        plt.show()


# =====================================================================
# 6. ANÁLISIS 4: SERIES TEMPORALES DIARIAS POR FRANJA HORARIA
# =====================================================================
def analisis_series_por_franja(finca, variables=None):
    """
    `variables`: lista de columnas a graficar. Si es None, usa todas las numéricas.
    Ejemplo:
        ['air_temperature', 'radiation', 'top_temperature_20cm', 'top_moisture_20cm',
         'bottom_ph_60cm', 'bottom_nitrogen_60cm', 'mppt_battery_voltage_V', 'mppt_panel_power_W']
    """
    configurar_estilo(1.4)

    df_fija = sin_columnas_geograficas(cargar_telemetria(finca, "fija", col_fecha="timestamp"))
    df_sist = sin_columnas_geograficas(cargar_telemetria(finca, "sistema", col_fecha="timestamp"))

    df_combinado = df_fija.join(df_sist, how="outer")
    df_combinado["Franja_Horaria"] = df_combinado.index.hour.map(asignar_franja)
    df_combinado["Date"] = df_combinado.index.date

    vars_plot = nombres_numericos(df_combinado) if variables is None else variables

    # Altura de 6 pulgadas por cada variable graficada
    fig, axes = plt.subplots(len(vars_plot), 1, figsize=(24, 6 * len(vars_plot)))
    if len(vars_plot) == 1:
        axes = [axes]

    for i, var in enumerate(vars_plot):
        ax = axes[i]

        # Promedios y desviaciones cruzando Fecha y Franja
        df_var_mean = df_combinado.groupby(["Date", "Franja_Horaria"])[var].mean().unstack()
        df_var_std = df_combinado.groupby(["Date", "Franja_Horaria"])[var].std().unstack().fillna(0)

        # Datetime para que Matplotlib lo interprete como eje temporal
        df_var_mean.index = pd.to_datetime(df_var_mean.index)
        df_var_std.index = pd.to_datetime(df_var_std.index)

        for franja in FRANJAS:
            if franja in df_var_mean.columns:
                mean_vals = df_var_mean[franja].dropna()
                std_vals = df_var_std[franja].loc[mean_vals.index]
                nombre_label = franja.split(".")[1].strip()

                ax.plot(mean_vals.index, mean_vals, label=f"Promedio {nombre_label}",
                        color=COLORES_FRANJA[franja], linewidth=2.5)
                ax.fill_between(mean_vals.index,
                                mean_vals - std_vals,
                                mean_vals + std_vals,
                                color=COLORES_FRANJA[franja], alpha=0.15)

        ax.set_title(f"Evolución Temporal por Franja Horaria: {var} - {finca.nombre}",
                     fontweight="bold", fontsize=18)
        ax.set_ylabel(f"Valor ({var})", fontsize=15)
        ax.legend(loc="upper right", fontsize=14, ncol=3)

        # Saltos fijos cada 10 días en el eje X
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=10))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.tick_params(axis="x", rotation=45, labelsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()

    dir_salida = crear_directorio(finca, "tendencias_por_franja_horaria")
    ruta = f"{dir_salida}/series_temporales_franjas_{finca.nombre}.png"
    fig.savefig(ruta, dpi=200, bbox_inches="tight")
    print(f"Archivo exportado en: {ruta}")

    plt.show()




In [ ]:
# =====================================================================
# 7. EJECUCIÓN
# =====================================================================
ANALISIS = {
    "boxplots_diarios": analisis_boxplots_diarios,
    "tendencias_diarias": analisis_tendencias_diarias,
    "violin_franjas": analisis_violin_franjas,
    "series_por_franja": analisis_series_por_franja,
}

analisis = None  # Lista de análisis a ejecutar (None = todos)
for finca in FINCAS:
    for nombre, funcion in ANALISIS.items():
        if analisis is None or nombre in analisis:
            print(f"\n {finca.nombre} | {nombre} ")
            funcion(finca)


 La_Orquidea | boxplots_diarios 
=== DATAFRAME: VARIABLES CON DATOS FALTANTES ===


,Variable,Datos_Faltantes,Porcentaje_%
20,medium_moisture41_pred,7747,100.0
21,medium_ecpw41_pred,7747,100.0
29,top_moisture20_pred,7747,100.0
30,top_ecpw20_pred,7747,100.0
31,conductivity_est_extratrees_20cm,7747,100.0
32,conductivity_est_extratrees_40cm,7747,100.0
33,conductivity_est_extratrees_60cm,7747,100.0
34,conductivity_est_randomforest_20cm,7747,100.0
35,conductivity_est_randomforest_40cm,7747,100.0
36,conductivity_est_randomforest_60cm,7747,100.0



=== DATAFRAME: MUESTRA DE OUTLIERS ===


,Fecha_Hora,Variable,Valor_Atipico,Voltaje_Bateria_V
0,2026-03-22 18:00,bottom_ph_60cm,5.53,12.32
1,2026-05-19 02:00,bottom_ph_60cm,7.14,11.73
2,2026-05-19 03:00,bottom_ph_60cm,7.14,11.72
3,2026-05-19 05:00,bottom_ph_60cm,7.17,11.69
4,2026-05-19 17:00,bottom_ph_60cm,7.14,12.32
5,2026-05-19 20:00,bottom_ph_60cm,7.19,12.26
6,2026-05-19 21:00,bottom_ph_60cm,7.16,12.24
7,2026-05-19 22:00,bottom_ph_60cm,7.20,12.23
8,2026-05-20 02:00,bottom_ph_60cm,7.14,12.18
9,2026-05-20 03:00,bottom_ph_60cm,7.14,12.20


Bloque 1-8 guardado en outputs/La_Orquidea/boxplots_diarios/boxplots_vars_1_a_8.png
Bloque 9-16 guardado en outputs/La_Orquidea/boxplots_diarios/boxplots_vars_9_a_16.png
Bloque 17-24 guardado en outputs/La_Orquidea/boxplots_diarios/boxplots_vars_17_a_24.png
